# 7 Wonders: symulacja i trening ONNX

Ten notebook uruchamia symulację w C#, zbiera plik wynikowy z logami ruchów i trenuje hierarchiczny model policy/value na faktycznym wektorze stanu.

Przepływ:
1. `GameConsole` eksportuje dane z `MoveLog.State`, `MoveLog.ActionMask` i `MoveLog.ActionIndex`.
2. Notebook wczytuje najnowszy plik `training_*.json`.
3. Model PyTorch uczy się i eksportuje `policy_network.onnx`.

In [1]:
EPOCHS = 50
GAMES_TRAIN = 500

In [2]:
from pathlib import Path
import importlib
import subprocess
import sys

repo_root = Path(r"c:/Users/kubeu/Kuba-dokumenty/Magisterka/7 Wonders")
game_console = repo_root / "GameConsole" / "GameConsole.csproj"
results_dir = repo_root / "GameConsole" / "Results"
encoding_dir = repo_root / "GameAI" / "Encoding"

sys.path.append(str(encoding_dir))

import game_training_pipeline as gtp
gtp = importlib.reload(gtp)

ActionSpace = gtp.ActionSpace
GameDataset = gtp.GameDataset
HierarchicalPolicyNetwork = gtp.HierarchicalPolicyNetwork
train_epoch = gtp.train_epoch
evaluate = gtp.evaluate

print("Repo root:", repo_root)
print("State vector size:", ActionSpace.STATE_VECTOR_SIZE)
print("Primary action size:", ActionSpace.TOTAL_PRIMARY_ACTIONS)

Repo root: c:\Users\kubeu\Kuba-dokumenty\Magisterka\7 Wonders
State vector size: 1903
Primary action size: 120


In [3]:
import torch
print(f"Czy CUDA działa? {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"Wykryta karta: {torch.cuda.get_device_name(0)}")
    print(f"Wersja CUDA w Torch: {torch.version.cuda}") # type: ignore

Czy CUDA działa? True
Wykryta karta: NVIDIA GeForce RTX 4050 Laptop GPU
Wersja CUDA w Torch: 12.4


In [ ]:
def run_simulation(seed: int = 12345, games: int = 20, agent1: str = "heuristic-personal", agent2: str = "mcts"):
    results_dir.mkdir(parents=True, exist_ok=True)
    command = [
        "dotnet", "run",
        "--project", str(game_console),
        "--",
        "export-data",
        "--seed", str(seed),
        "--games", str(games),
        "--agent1", agent1,
        "--agent2", agent2,
    ]
    subprocess.run(command, cwd=repo_root, check=True)

run_simulation(seed=1, games=GAMES_TRAIN, agent1="heuristic-personal", agent2="onnx3")
print("Simulation finished.")

Simulation finished.


## Uwagi na temat pamięci: --minimal-logs dla dużych zbiorów

Podczas generowania danych treningowych dla 1000+ gier można napotkać problemy z pamięcią. Flag `--minimal-logs` usuwa z pliku:
- Detale kart (Karta, KartaCudu)
- Surowce graczy
- Opisy ruchów (TypRuchu, KartyDoWyboru)
- Decyzje pośrednie (subdecisions)

Zmniejszenie: ~11x (z 44 MB na 4 MB na 10 gier).

### Użycie:
```bash
dotnet run --project GameConsole/GameConsole.csproj -- export-data \
  --games 1000 --seed 12345 --agent1 heuristic-double --agent2 mcts \
  --minimal-logs
```

Plik zostanie zapisany jako `training_..._minimal.json`.

In [4]:
from torch.utils.data import DataLoader, random_split
import torch

training_files = sorted(results_dir.glob("training_*.json"))
if not training_files:
    raise FileNotFoundError(f"No training_*.json files found in {results_dir}")

latest_file = max(training_files, key=lambda p: p.stat().st_mtime)
print("Using dataset:", latest_file)

dataset = GameDataset(str(latest_file), normalize=True, validate_shapes=True)
train_size = max(1, int(len(dataset) * 0.9))
val_size = max(1, len(dataset) - train_size)
if train_size + val_size > len(dataset):
    val_size = len(dataset) - train_size

train_dataset, val_dataset = random_split(dataset, [train_size, val_size], generator=torch.Generator().manual_seed(42))
train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=32, shuffle=False) if len(val_dataset) > 0 else None

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = HierarchicalPolicyNetwork(state_dim=ActionSpace.STATE_VECTOR_SIZE, hidden_dim=256, dropout=0.1).to(device)
optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)

for epoch in range(EPOCHS):
    train_stats = train_epoch(model, train_loader, optimizer, device)
    if val_loader is not None:
        val_stats = evaluate(model, val_loader, device)
        print(f"epoch={epoch + 1} train={train_stats['total_loss']:.4f} val={val_stats['total_loss']:.4f}")
    else:
        print(f"epoch={epoch + 1} train={train_stats['total_loss']:.4f}")

2026-05-11 14:16:45,497 - game_training_pipeline - INFO - Loading dataset from c:\Users\kubeu\Kuba-dokumenty\Magisterka\7 Wonders\GameConsole\Results\training_20260511_140717_onnx3_vs_heuristic-double_500_games_minimal.json


Using dataset: c:\Users\kubeu\Kuba-dokumenty\Magisterka\7 Wonders\GameConsole\Results\training_20260511_140717_onnx3_vs_heuristic-double_500_games_minimal.json


2026-05-11 14:16:58,656 - game_training_pipeline - INFO - Loaded 29123 valid samples
2026-05-11 14:16:59,041 - game_training_pipeline - INFO - State normalized: mean=0.0476, std=0.1109


epoch=1 train=1.2334 val=1.0478
epoch=2 train=0.8808 val=0.9343
epoch=3 train=0.6338 val=0.9611
epoch=4 train=0.4390 val=1.0754
epoch=5 train=0.2916 val=1.4058
epoch=6 train=0.2078 val=1.6625
epoch=7 train=0.1628 val=1.9765
epoch=8 train=0.1388 val=2.2067
epoch=9 train=0.1293 val=2.4763
epoch=10 train=0.1080 val=2.7390
epoch=11 train=0.1056 val=2.7540
epoch=12 train=0.1059 val=3.0849
epoch=13 train=0.1080 val=3.0569
epoch=14 train=0.0918 val=3.5239
epoch=15 train=0.0992 val=3.5337
epoch=16 train=0.0931 val=3.6783
epoch=17 train=0.0983 val=4.3049
epoch=18 train=0.1007 val=4.1852
epoch=19 train=0.0887 val=4.3965
epoch=20 train=0.0897 val=4.4424
epoch=21 train=0.0845 val=4.4612
epoch=22 train=0.0877 val=4.6794
epoch=23 train=0.0896 val=4.9475
epoch=24 train=0.0893 val=5.0628
epoch=25 train=0.1013 val=5.1585
epoch=26 train=0.0809 val=5.3649
epoch=27 train=0.0889 val=6.0225
epoch=28 train=0.0873 val=5.9124
epoch=29 train=0.0850 val=6.0101
epoch=30 train=0.0900 val=6.2679
epoch=31 train=0.09

In [5]:
onnx_path = encoding_dir / f"onnx_models/policy_network_{EPOCHS}_{GAMES_TRAIN}.onnx"
model.onnx_export(str(onnx_path), validate=True)
print("Exported:", onnx_path)

batch = next(iter(train_loader))
state = batch['state'].to(device)
action_mask = batch['action_mask'].to(device)
outputs = model(state, action_mask=action_mask)
print("policy shape:", outputs['policy_masked_logits'].shape)
print("value shape:", outputs['value'].shape)

2026-05-11 14:25:08,779 - game_training_pipeline - INFO - Exporting model to ONNX format: c:\Users\kubeu\Kuba-dokumenty\Magisterka\7 Wonders\GameAI\Encoding\onnx_models\policy_network_50_500.onnx
c:\Users\kubeu\Kuba-dokumenty\Magisterka\7 Wonders\GameAI\Encoding\game_training_pipeline.py:140: TracerWarning: Converting a tensor to a Python boolean might cause the trace to be incorrect. We can't record the data flow of Python values, so this value will be treated as a constant in the future. This means that the trace might not generalize to other inputs!
  assert action_mask.shape == policy_logits.shape, \
2026-05-11 14:25:09,040 - game_training_pipeline - INFO - ✓ Model exported successfully: c:\Users\kubeu\Kuba-dokumenty\Magisterka\7 Wonders\GameAI\Encoding\onnx_models\policy_network_50_500.onnx
2026-05-11 14:25:09,041 - game_training_pipeline - INFO - Validating ONNX model: c:\Users\kubeu\Kuba-dokumenty\Magisterka\7 Wonders\GameAI\Encoding\onnx_models\policy_network_50_500.onnx
2026-0

Exported: c:\Users\kubeu\Kuba-dokumenty\Magisterka\7 Wonders\GameAI\Encoding\onnx_models\policy_network_50_500.onnx
policy shape: torch.Size([32, 120])
value shape: torch.Size([32, 1])
